# 11 — VSL400: trích xuất pose RTMPose-L WholeBody cho 50 từ, 3 góc quay

Notebook **độc lập** này làm việc với bộ VSL400 **đã giải nén trên Google Drive** (`front_view/`, `left_view/`, `right_view/`, ba file JSON và `gloss.csv`):

1. **Manifest + split cho toàn bộ 400 từ.** `ss-prepare all` đọc metadata 3 góc, kiểm tra cấu trúc và file, rồi chia **theo người ký** (signer-disjoint) theo tỉ lệ 80/10/10. Với 28 người ký, kết quả là 22 / 3 / 3. VSL400 không có split chính thức, nên đây là quy trình của dự án: tìm kiếm có seed 42, luôn cho cùng một kết quả. Nếu đã chạy notebook 00, kết quả trên Drive được dùng lại.
2. **Chọn 50 từ** trên split đó: xếp theo **số lần ký trong tập train**, và chỉ giữ từ có mặt ở cả train/validation/test. Split **không bao giờ bị chia lại** trên tập con. Cũng có thể liệt kê chính xác `GLOSS_IDS` muốn dùng.
3. **Chép đúng các video cần dùng** (cả 3 góc) từ Drive vào `/content` và kiểm tra lại bằng ffprobe (25 fps, 1080×1080, đủ 3 góc cho mỗi lần ký).
4. **RTMDet-M + RTMPose-L 384×288** chạy trên mọi frame của từng video, lưu 133 keypoint thô cho mỗi clip. Có thể resume và chia nhiều runtime.
5. **Tensor đồ thị** `[64, 75, 7]` cho graph encoder.

VSL400 có **tên tiếng Việt** cho từng từ (`gloss.csv`); danh sách 50 từ được in ra và lưu trong `selection.json`.

Trước khi chạy: **Runtime → Change runtime type → T4 GPU** (hoặc L4/A100). Chỉ dùng VSL400 theo đúng Data Usage Agreement của tác giả.

In [ ]:
PROJECT_GIT_URL = 'https://github.com/stillthethrone/silent-signal.git'
PROJECT_GIT_REF = 'feat/multi-vsl-rtmpose-extraction'  # @param {type:'string'}

# Thư mục VSL400 đã giải nén trên Drive (chứa front_view/, left_view/, right_view/, *.json, gloss.csv).
DRIVE_DATASET_ROOT = '/content/drive/MyDrive/VSL400'  # @param {type:'string'}
CLASS_COUNT = 50  # @param {type:'integer'}
# Để trống để chọn tự động; điền gloss_id (chuỗi) để lấy đúng các từ đó theo thứ tự.
GLOSS_IDS = []
# Dùng lại manifest/split toàn bộ 400 từ đã có trên Drive (từ notebook 00 hoặc lần chạy trước).
REUSE_FULL_PREPARATION = True  # @param {type:'boolean'}
VALIDATION_LEVEL = 'probe'  # @param ['metadata', 'probe']
COPY_WORKERS = 8  # @param {type:'integer'}

RUN_PILOT = True  # @param {type:'boolean'}
PILOT_LIMIT = 20  # @param {type:'integer'}
RUN_FULL_EXTRACTION = True  # @param {type:'boolean'}
NUM_SHARDS = 1  # @param {type:'integer'}
SHARD_INDEX = 0  # @param {type:'integer'}
RUN_GRAPH_PREPARATION = True  # @param {type:'boolean'}
OVERWRITE = False  # @param {type:'boolean'}

GLOSS_IDS = [str(value) for value in GLOSS_IDS]
if not GLOSS_IDS and CLASS_COUNT < 2:
    raise ValueError('CLASS_COUNT phải từ 2 trở lên.')
if PILOT_LIMIT < 1 or COPY_WORKERS < 1:
    raise ValueError('PILOT_LIMIT và COPY_WORKERS phải lớn hơn 0.')
if NUM_SHARDS < 1 or not 0 <= SHARD_INDEX < NUM_SHARDS:
    raise ValueError('Cần 0 <= SHARD_INDEX < NUM_SHARDS.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DRIVE_DATASET_ROOT = Path(DRIVE_DATASET_ROOT)
PROJECT_ROOT = Path('/content/silent-signal')
LOCAL_DATASET_ROOT = Path('/content/VSL400_subset')

# Cùng vị trí với notebook 00 để dùng lại manifest/split toàn bộ dataset.
RESULTS_ROOT = Path('/content/drive/MyDrive/silent-signal-results/vsl400')
FULL_MANIFEST = RESULTS_ROOT / 'manifests/vsl400.parquet'
FULL_SPLIT = RESULTS_ROOT / 'splits/vsl400_signer_split.json'
FULL_CONFIG = RESULTS_ROOT / 'configs/vsl400.full.yaml'

SUBSET_NAME = 'custom_' + '_'.join(GLOSS_IDS) if GLOSS_IDS else f'top{CLASS_COUNT}'
SUBSET_ROOT = RESULTS_ROOT / 'subsets' / f'{SUBSET_NAME}_three_view'
PREPARED_ROOT = SUBSET_ROOT / 'prepared'
MANIFEST = PREPARED_ROOT / 'manifest.csv'
SELECTION = PREPARED_ROOT / 'selection.json'
SUBSET_CONFIG = PREPARED_ROOT / 'vsl400.subset.yaml'
POSE_ROOT = SUBSET_ROOT / 'pose/rtmpose_l_coco_wholebody_384x288'
POSE_OUTPUT_ROOT = POSE_ROOT / 'raw'
PINNED_CONFIG = POSE_ROOT / 'provenance/rtmpose-colab-pinned.yaml'
GRAPH_ROOT = SUBSET_ROOT / 'graph/coco_wholebody_75_v1_t64'

POSE_ENV_ROOT = Path('/content/pose-env')
MMPOSE_ROOT = Path('/content/mmpose-v1.3.2')
MODEL_ROOT = Path('/content/drive/MyDrive/silent-signal-models/openmmlab')
POSE_CHECKPOINT = MODEL_ROOT / 'rtmpose-l-wholebody-384x288.pth'
DET_CHECKPOINT = MODEL_ROOT / 'rtmdet-m-person.pth'

if not DRIVE_DATASET_ROOT.is_dir():
    raise FileNotFoundError(f'Không tìm thấy VSL400 trên Drive: {DRIVE_DATASET_ROOT}')
layout = {
    'front': (('front_view', 'front_view.json'), ('cam_1', 'cam_1.json')),
    'left': (('left_view', 'left_view.json'), ('cam_2', 'cam_2.json')),
    'right': (('right_view', 'right_view.json'), ('cam_3', 'cam_3.json')),
}
for view, candidates in layout.items():
    if not any((DRIVE_DATASET_ROOT / folder).is_dir() and (DRIVE_DATASET_ROOT / meta).is_file()
               for folder, meta in candidates):
        raise FileNotFoundError(f'Thiếu thư mục hoặc JSON của góc {view}: {candidates}')
if not (DRIVE_DATASET_ROOT / 'gloss.csv').is_file():
    print('Cảnh báo: không thấy gloss.csv; tên từ sẽ lấy từ metadata JSON.')

for path in (LOCAL_DATASET_ROOT, PREPARED_ROOT, POSE_OUTPUT_ROOT, PINNED_CONFIG.parent,
             GRAPH_ROOT, MODEL_ROOT, FULL_CONFIG.parent):
    path.mkdir(parents=True, exist_ok=True)
print('VSL400 trên Drive:', DRIVE_DATASET_ROOT)
print('Kết quả tập con:', SUBSET_ROOT)

## Lấy mã nguồn

Clone đúng nhánh `PROJECT_GIT_REF` của dự án. Nếu nhánh chưa được push lên GitHub, cell sẽ dừng.

In [ ]:
import os
import subprocess
import sys
import time

def run(command, cwd=None, env=None):
    command = [str(part) for part in command]
    print('+', ' '.join(command), flush=True)
    started = time.perf_counter()
    process = subprocess.Popen(
        command, cwd=cwd, env=env, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    try:
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end='', flush=True)
        code = process.wait()
    except KeyboardInterrupt:
        process.terminate()
        process.wait()
        raise
    print(f'[{(time.perf_counter() - started) / 60:.1f} min]', flush=True)
    if code:
        raise subprocess.CalledProcessError(code, command)

if not PROJECT_ROOT.exists():
    run(['git', 'clone', '--depth', '1', '--branch', PROJECT_GIT_REF, PROJECT_GIT_URL, PROJECT_ROOT])
else:
    run(['git', '-C', PROJECT_ROOT, 'fetch', '--depth', '1', 'origin', PROJECT_GIT_REF])
    run(['git', '-C', PROJECT_ROOT, 'checkout', '--detach', 'FETCH_HEAD'])

PROJECT_COMMIT = subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
required_files = [
    PROJECT_ROOT / 'src/silent_signal/cli/select_classes.py',
    PROJECT_ROOT / 'configs/dataset/vsl400.yaml',
    PROJECT_ROOT / 'configs/pose/rtmpose.yaml',
    PROJECT_ROOT / 'configs/preprocessing/coco_wholebody_75_t64.yaml',
]
missing = [str(path) for path in required_files if not path.is_file()]
if missing:
    raise RuntimeError('Nhánh này thiếu file cần thiết: ' + ', '.join(missing))
print('Project commit:', PROJECT_COMMIT)

## Môi trường RTMPose cố định

Tạo Python 3.11 riêng trong `/content/pose-env` với bộ phiên bản đã kiểm chứng trên Colab: NumPy 1.26.4, PyTorch 2.1.0 + CUDA 12.1, MMCV 2.1.0, MMDetection 3.2.0, MMPose 1.3.2. Kernel Colab không import các thư viện này nên không bị xung đột ABI. Nếu marker môi trường còn trong runtime hiện tại, phần cài nặng được bỏ qua.

In [ ]:
import shutil

run(['nvidia-smi'])
print(f"/content còn trống: {shutil.disk_usage('/content').free / 1024**3:.1f} GiB")
if shutil.which('ffprobe') is None:
    run(['apt-get', 'install', '-y', '-qq', 'ffmpeg'])
run([sys.executable, '-m', 'pip', 'install', '--quiet', 'uv'])
run(['uv', 'python', 'install', '3.11'])
if not (POSE_ENV_ROOT / 'bin/python').is_file():
    run(['uv', 'venv', POSE_ENV_ROOT, '--python', '3.11', '--seed'])

POSE_PY = str(POSE_ENV_ROOT / 'bin/python')
PREPARE_CLI = str(POSE_ENV_ROOT / 'bin/ss-prepare')
SELECT_CLI = str(POSE_ENV_ROOT / 'bin/ss-select-classes')
POSE_CLI = str(POSE_ENV_ROOT / 'bin/ss-extract-pose')
GRAPH_CLI = str(POSE_ENV_ROOT / 'bin/ss-prepare-pose-graph')
ENV_MARKER = POSE_ENV_ROOT / '.silent-signal-rtmpose-v1.ready'

if not MMPOSE_ROOT.exists():
    run(['git', 'clone', '--depth', '1', '--branch', 'v1.3.2', 'https://github.com/open-mmlab/mmpose.git', MMPOSE_ROOT])

if not ENV_MARKER.is_file():
    run([POSE_PY, '-m', 'pip', 'install', 'pip==24.3.1', 'setuptools==75.6.0', 'wheel==0.45.1'])
    run([POSE_PY, '-m', 'pip', 'install', 'numpy==1.26.4'])
    run([POSE_PY, '-m', 'pip', 'install', '--no-build-isolation', 'chumpy==0.70'])
    run([POSE_PY, '-m', 'pip', 'install', 'torch==2.1.0', 'torchvision==0.16.0',
         '--index-url', 'https://download.pytorch.org/whl/cu121',
         '--extra-index-url', 'https://pypi.org/simple'])
    run([POSE_PY, '-m', 'pip', 'install', 'mmengine==0.10.7'])
    run([POSE_PY, '-m', 'pip', 'install', 'mmcv==2.1.0',
         '-f', 'https://download.openmmlab.com/mmcv/dist/cu121/torch2.1/index.html'])
    run([POSE_PY, '-m', 'pip', 'install', 'mmdet==3.2.0'])
    run([POSE_PY, '-m', 'pip', 'install', '-e', MMPOSE_ROOT])

run([POSE_PY, '-m', 'pip', 'install', '-e', PROJECT_ROOT])
environment_check = '''
import mmcv, mmdet, mmpose, numpy as np, torch
from mmcv.ops import nms
assert torch.cuda.is_available(), 'CUDA không khả dụng trong pose-env.'
print('GPU:', torch.cuda.get_device_name(0))
print('NumPy', np.__version__, '| torch', torch.__version__, '| MMCV', mmcv.__version__, '| MMPose', mmpose.__version__)
'''
run([POSE_PY, '-c', environment_check])
ENV_MARKER.write_text(PROJECT_COMMIT + '\n', encoding='utf-8')

PROCESS_ENV = os.environ.copy()
PROCESS_ENV.update({
    'PYTHONUNBUFFERED': '1',
    'MPLBACKEND': 'Agg',
    'MMPOSE_ROOT': str(MMPOSE_ROOT),
    'RTMPOSE_L_WHOLEBODY_CHECKPOINT': str(POSE_CHECKPOINT),
    'RTMDET_M_PERSON_CHECKPOINT': str(DET_CHECKPOINT),
})

## Bước 1 — Manifest và split cho toàn bộ 400 từ

Split được tạo trên **toàn bộ dataset** trước khi chọn từ, nên 50 từ được chọn không ảnh hưởng tới việc phân người ký. Trên Drive, bước kiểm tra `metadata` cần đọc thông tin của khoảng 74.000 file, có thể mất vài chục phút ở lần chạy đầu. Các lần sau dùng lại kết quả (`REUSE_FULL_PREPARATION=True`).

In [ ]:
import json
import yaml

if REUSE_FULL_PREPARATION and FULL_MANIFEST.is_file() and FULL_SPLIT.is_file():
    print('Dùng lại manifest/split đã có:', FULL_MANIFEST)
else:
    full_config = yaml.safe_load((PROJECT_ROOT / 'configs/dataset/vsl400.yaml').read_text(encoding='utf-8'))
    full_config['dataset']['root'] = str(DRIVE_DATASET_ROOT)
    full_config['outputs'] = {
        'manifest_csv': str(RESULTS_ROOT / 'manifests/vsl400.csv'),
        'manifest_parquet': str(FULL_MANIFEST),
        'labels': str(RESULTS_ROOT / 'labels/vsl400_labels.json'),
        'split': str(FULL_SPLIT),
        'report': str(RESULTS_ROOT / 'reports/metadata_validation_report.json'),
        'invalid_records': str(RESULTS_ROOT / 'reports/metadata_invalid_records.csv'),
    }
    FULL_CONFIG.write_text(yaml.safe_dump(full_config, sort_keys=False, allow_unicode=True), encoding='utf-8')
    run([PREPARE_CLI, 'all', '--config', FULL_CONFIG, '--level', 'metadata', '--workers', 8],
        cwd=PROJECT_ROOT, env=PROCESS_ENV)
    if not FULL_SPLIT.is_file():
        raise RuntimeError('Không tạo được split; xem reports/metadata_validation_report.json trên Drive.')

full_split = json.loads(FULL_SPLIT.read_text(encoding='utf-8'))
print('Chia theo người ký (toàn bộ 400 từ):')
for name in ('train', 'validation', 'test'):
    print(f"  {name:<10} {full_split['signer_counts'][name]:>2} người ký  "
          f"{full_split['instance_counts'][name]:>6} lần ký  {full_split['clip_counts'][name]:>6} video  "
          f"signer={full_split['signer_ids'][name]}")

## Bước 2 — Chọn 50 từ

Mặc định lấy các từ có **nhiều lần ký nhất trong tập train** và có mặt ở cả 3 split; nếu bằng nhau thì giữ thứ tự gloss gốc. Validation/test **không** được dùng để xếp hạng. `class_index` mới chạy từ 0 theo thứ hạng; `gloss_id` và tên tiếng Việt được giữ nguyên.

In [ ]:
command = [SELECT_CLI, '--manifest', FULL_MANIFEST, '--output-root', PREPARED_ROOT,
           '--classes', CLASS_COUNT, '--dataset-name', 'vsl400']
for gloss_id in GLOSS_IDS:
    command += ['--gloss-id', gloss_id]
run(command, env=PROCESS_ENV)

selection = json.loads(SELECTION.read_text(encoding='utf-8'))
REQUIRED_VIDEOS = [line for line in (PREPARED_ROOT / 'required_videos.txt').read_text(encoding='utf-8').splitlines() if line]
print(f"\n{selection['class_count']} từ | góc quay: {selection['views']} | tổng {len(REQUIRED_VIDEOS):,} video")
for name in ('train', 'validation', 'test'):
    print(f"  {name:<10} {len(selection['signers'][name]):>2} người ký  "
          f"{selection['instances'][name]:>5} lần ký  {selection['clips'][name]:>5} video")
print(f"\n{'rank':>4} {'class':>5} {'gloss_id':>8}  {'từ':<24} {'train':>5} {'val':>4} {'test':>4}")
for item in selection['classes']:
    counts = item['instances']
    print(f"{item['rank']:>4} {item['class_index']:>5} {item['gloss_id']:>8}  {item['gloss_name'][:24]:<24} "
          f"{counts['train']:>5} {counts['validation']:>4} {counts['test']:>4}")

## Bước 3 — Chép video cần dùng vào runtime

Chỉ chép các video của 50 từ (cả 3 góc), giữ nguyên đường dẫn tương đối như `front_view/000123.mp4`. File đã có với đúng kích thước sẽ được bỏ qua; mỗi file được ghi vào `.part` trước rồi mới đổi tên. Video trong `/content` sẽ mất khi runtime reset; khi đó chỉ cần chạy lại cell này.

In [ ]:
import concurrent.futures

def copy_one(relative):
    source = DRIVE_DATASET_ROOT / relative
    destination = LOCAL_DATASET_ROOT / relative
    if not source.is_file():
        return relative, 'missing'
    size = source.stat().st_size
    if destination.is_file() and destination.stat().st_size == size:
        return relative, 'kept'
    destination.parent.mkdir(parents=True, exist_ok=True)
    partial = destination.with_name(f'.{destination.name}.part')
    shutil.copyfile(source, partial)
    if partial.stat().st_size != size:
        partial.unlink(missing_ok=True)
        return relative, 'size mismatch'
    partial.replace(destination)
    return relative, 'copied'

print(f"/content còn trống: {shutil.disk_usage('/content').free / 1024**3:.1f} GiB")
status = {}
with concurrent.futures.ThreadPoolExecutor(max_workers=COPY_WORKERS) as pool:
    futures = [pool.submit(copy_one, relative) for relative in REQUIRED_VIDEOS]
    for position, future in enumerate(concurrent.futures.as_completed(futures), start=1):
        relative, result = future.result()
        status[relative] = result
        if position % 250 == 0 or position == len(futures):
            counts = {key: list(status.values()).count(key) for key in ('copied', 'kept', 'missing', 'size mismatch')}
            print(f'[copy] {position:,}/{len(futures):,} {counts}', flush=True)
problems = sorted(relative for relative, result in status.items() if result in {'missing', 'size mismatch'})
if problems:
    raise RuntimeError(f'{len(problems)} video lỗi hoặc thiếu trên Drive; ví dụ {problems[:3]}')
size_gib = sum((LOCAL_DATASET_ROOT / relative).stat().st_size for relative in REQUIRED_VIDEOS) / 1024**3
print(f'Đủ {len(REQUIRED_VIDEOS):,} video ({size_gib:.1f} GiB) trong {LOCAL_DATASET_ROOT}')

## Bước 4 — Kiểm tra lại tập con trên bản chép

`ss-prepare validate` chạy trên bản chép trong `/content` với số lượng kỳ vọng của đúng tập con: đủ 3 góc cho mỗi lần ký, cùng người ký và nhãn, file không rỗng. Với `VALIDATION_LEVEL='probe'`, lệnh này còn đọc header bằng ffprobe để kiểm tra 25 fps, 1080×1080 và số frame. Manifest trên Drive được cập nhật thêm các thông số đo được; RTMPose sau đó từ chối cache nếu số frame hoặc kích thước giải mã được lệch so với manifest.

In [ ]:
subset_config = yaml.safe_load((PROJECT_ROOT / 'configs/dataset/vsl400.yaml').read_text(encoding='utf-8'))
subset_config['dataset']['root'] = str(LOCAL_DATASET_ROOT)
subset_config['expected'].update({
    'clips': sum(selection['clips'].values()),
    'glosses': selection['class_count'],
    'signers': len({signer for signers in selection['signers'].values() for signer in signers}),
})
subset_config['outputs'] = {
    'manifest_csv': str(MANIFEST),
    'manifest_parquet': str(PREPARED_ROOT / 'manifest.parquet'),
    'labels': str(PREPARED_ROOT / 'labels.json'),
    'split': str(PREPARED_ROOT / 'unused_split.json'),
    'report': str(PREPARED_ROOT / 'validation_report.json'),
    'invalid_records': str(PREPARED_ROOT / 'invalid_records.csv'),
}
SUBSET_CONFIG.write_text(yaml.safe_dump(subset_config, sort_keys=False, allow_unicode=True), encoding='utf-8')
run([PREPARE_CLI, 'validate', '--config', SUBSET_CONFIG, '--manifest', PREPARED_ROOT / 'manifest.parquet',
     '--level', VALIDATION_LEVEL, '--workers', 8], cwd=PROJECT_ROOT, env=PROCESS_ENV)
report = json.loads((PREPARED_ROOT / 'validation_report.json').read_text(encoding='utf-8'))
print('Passed:', report['passed'], '| issues:', report['issue_counts'])

## Bước 5 — Tải model và cố định SHA-256

Checkpoint được giữ trên Drive để dùng lại giữa các lần chạy và dùng chung với notebook 10. Notebook tính SHA-256 thực tế rồi ghi vào một bản config đã pin; lúc trích xuất, extractor từ chối chạy nếu checkpoint khác bản đã pin.

In [ ]:
POSE_URL = ('https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/'
            'rtmpose-l_simcc-coco-wholebody_pt-aic-coco_270e-384x288-eaeb96c8_20230125.pth')
DET_URL = ('https://download.openmmlab.com/mmpose/v1/projects/rtmpose/'
           'rtmdet_m_8xb32-100e_coco-obj365-person-235e8209.pth')

for url, destination in ((POSE_URL, POSE_CHECKPOINT), (DET_URL, DET_CHECKPOINT)):
    if destination.is_file():
        print('Đã có:', destination)
        continue
    partial = Path(str(destination) + '.part')
    run(['wget', '-c', url, '-O', partial])
    if partial.stat().st_size == 0:
        raise RuntimeError(f'Checkpoint tải về rỗng: {partial}')
    partial.replace(destination)

UNPINNED_LOCK = PINNED_CONFIG.parent / 'unverified-hashes.lock.json'
PINNED_LOCK = PINNED_CONFIG.parent / 'rtmpose-colab-pinned.lock.json'
SOURCE_POSE_CONFIG = PROJECT_ROOT / 'configs/pose/rtmpose.yaml'
run([POSE_CLI, 'verify', '--config', SOURCE_POSE_CONFIG, '--write-lock', UNPINNED_LOCK], env=PROCESS_ENV)
hashes = json.loads(UNPINNED_LOCK.read_text(encoding='utf-8'))
pose_config = yaml.safe_load(SOURCE_POSE_CONFIG.read_text(encoding='utf-8'))
pose_config['extractor']['pose_model']['checkpoint_sha256'] = hashes['pose_model']['checkpoint_sha256']
pose_config['extractor']['detector']['checkpoint_sha256'] = hashes['detector']['checkpoint_sha256']
PINNED_CONFIG.write_text(yaml.safe_dump(pose_config, sort_keys=False, allow_unicode=True), encoding='utf-8')
run([POSE_CLI, 'verify', '--config', PINNED_CONFIG, '--write-lock', PINNED_LOCK], env=PROCESS_ENV)
print('Pinned config:', PINNED_CONFIG)

## Bước 6 — Pilot

Chạy thử trên `PILOT_LIMIT` clip train đầu tiên (xếp theo `sample_id`). Cache được ghi thẳng vào thư mục `raw`; lượt chạy đầy đủ sẽ kiểm tra và dùng lại các cache hợp lệ này. Nếu một cache cũ không khớp model hoặc video nguồn, CLI dừng lại chứ không âm thầm ghi đè.

In [ ]:
def run_pose(command, report_path, label):
    try:
        run(command, env=PROCESS_ENV)
    except subprocess.CalledProcessError as exc:
        # Exit code 1 means some clips failed and the report lists them; others are setup errors.
        if exc.returncode != 1 or not report_path.is_file():
            raise
    report = json.loads(report_path.read_text(encoding='utf-8'))
    print(json.dumps({key: value for key, value in report.items() if key != 'failures'}, ensure_ascii=False, indent=2))
    if report['failed']:
        print(json.dumps(report['failures'][:20], ensure_ascii=False, indent=2))
        raise RuntimeError(f'{label}: {report["failed"]} video lỗi; xem {report_path}.')
    return report

def extract_command(report_path, *extra):
    command = [POSE_CLI, 'extract', '--config', PINNED_CONFIG, '--manifest', MANIFEST,
               '--dataset-root', LOCAL_DATASET_ROOT, '--output-root', POSE_OUTPUT_ROOT,
               '--report', report_path, '--device', 'cuda:0', *extra]
    return command + (['--overwrite'] if OVERWRITE else [])

PILOT_REPORT = POSE_ROOT / f'pilot_train_{PILOT_LIMIT}.json'
if RUN_PILOT:
    started = time.perf_counter()
    run_pose(extract_command(PILOT_REPORT, '--split', 'train', '--limit', PILOT_LIMIT, '--progress-every', 1),
             PILOT_REPORT, 'Pilot')
    seconds = (time.perf_counter() - started) / PILOT_LIMIT
    print(f'~{seconds:.1f} s/clip (gồm cả nạp model) → ước tính {seconds * len(REQUIRED_VIDEOS) / NUM_SHARDS / 3600:.1f} giờ cho toàn bộ.')
else:
    print('RUN_PILOT=False — bỏ qua pilot.')

## Bước 7 — Trích xuất toàn bộ train / validation / test

Không dùng `--split`: cả 3 split và cả 3 góc đều cần pose. Để chạy song song trên nhiều runtime Colab, mọi runtime đặt cùng `NUM_SHARDS` và mỗi runtime một `SHARD_INDEX` khác nhau; mỗi clip được gán tất định vào đúng một shard. Mỗi runtime phải chạy lại bước 3 (chép video) trước. Nếu runtime bị ngắt, chỉ cần chạy lại: các cache đã hợp lệ được bỏ qua.

In [ ]:
FULL_REPORT = POSE_ROOT / f'full_shard_{SHARD_INDEX:03d}_of_{NUM_SHARDS:03d}.json'
if RUN_FULL_EXTRACTION:
    run_pose(extract_command(FULL_REPORT, '--num-shards', NUM_SHARDS, '--shard-index', SHARD_INDEX,
                             '--continue-on-error', '--progress-every', 10),
             FULL_REPORT, 'Full extraction')
else:
    print('RUN_FULL_EXTRACTION=False — chưa chạy toàn bộ.')

## Bước 8 — Tạo tensor đồ thị `[64, 75, 7]`

Chạy sau khi **mọi shard** đã xong. Bước này không đọc video và không cần GPU: chọn 75 khớp (13 thân, 21×2 bàn tay, 20 mặt), nội suy khoảng trống ≤ 3 frame, chuẩn hóa theo vai/hông, lấy mẫu đều 64 frame, rồi tạo 7 kênh (x, y, confidence, vận tốc x/y, vector xương x/y). Mỗi góc là một mẫu riêng; gom theo `instance_id` khi dùng mô hình đa góc.

In [ ]:
import hashlib

if RUN_GRAPH_PREPARATION:
    full_reports = sorted(POSE_ROOT.glob(f'full_shard_*_of_{NUM_SHARDS:03d}.json'))
    if len(full_reports) != NUM_SHARDS:
        raise RuntimeError(f'Mới có {len(full_reports)}/{NUM_SHARDS} report shard; chạy đủ các shard trước.')
    fingerprints = {json.loads(path.read_text(encoding='utf-8'))['extractor_fingerprint'] for path in full_reports}
    if len(fingerprints) != 1:
        raise RuntimeError('Các shard dùng extractor khác nhau.')
    graph_config = yaml.safe_load((PROJECT_ROOT / 'configs/preprocessing/coco_wholebody_75_t64.yaml').read_text(encoding='utf-8'))
    graph_config['expected'] = {
        'manifest_sha256': hashlib.sha256(MANIFEST.read_bytes()).hexdigest(),
        'extractor_fingerprint': fingerprints.pop(),
        'clips': sum(selection['clips'].values()),
        'classes': selection['class_count'],
        'splits': selection['clips'],
    }
    GRAPH_CONFIG = GRAPH_ROOT / 'graph-config.pinned.yaml'
    GRAPH_CONFIG.write_text(yaml.safe_dump(graph_config, sort_keys=False), encoding='utf-8')
    GRAPH_REPORT = GRAPH_ROOT / 'graph_preparation_report.json'
    command = [GRAPH_CLI, '--config', GRAPH_CONFIG, '--manifest', MANIFEST,
               '--pose-root', POSE_OUTPUT_ROOT, '--output-root', GRAPH_ROOT / 'cache',
               '--report', GRAPH_REPORT, '--progress-every', 250]
    run(command + (['--overwrite'] if OVERWRITE else []), env=PROCESS_ENV)
    graph_report = json.loads(GRAPH_REPORT.read_text(encoding='utf-8'))
    print(json.dumps({key: value for key, value in graph_report.items() if key != 'failures'}, ensure_ascii=False, indent=2))
else:
    print('RUN_GRAPH_PREPARATION=False — bỏ qua.')

## Kết quả trên Drive

`MyDrive/silent-signal-results/vsl400/`

- `manifests/`, `splits/vsl400_signer_split.json`, `labels/`, `reports/`: toàn bộ 400 từ (dùng chung với notebook 00).
- `subsets/<subset>_three_view/prepared/`:
  - `selection.json`: **danh sách từ** (rank, `class_index`, `gloss_id`, tên tiếng Việt, số lần ký và số video mỗi split) cùng SHA-256 của manifest nguồn;
  - `manifest.csv|parquet`: một dòng cho mỗi video, gồm `view`, `instance_id` (chung cho 3 góc), `split`, `class_index` và thông số ffprobe;
  - `labels.json`, `required_videos.txt`, `validation_report.json`.
- `subsets/<subset>_three_view/pose/.../raw/`: pose thô 133 điểm cho mỗi video, lưu dạng NPZ không dùng pickle, có thể resume; kèm `provenance/` (SHA-256 của model, phiên bản thư viện).
- `subsets/<subset>_three_view/graph/.../cache/`: tensor `[64, 75, 7]` kèm mask.

Video nguồn chỉ được chép tạm vào `/content`. Không commit dữ liệu, manifest sinh ra hoặc cache vào Git.